In [19]:
import pandas as pd
from img2table.document import PDF
from img2table.ocr import TesseractOCR
import os
import re

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS
# ==========================================
# Ajusta estas rutas a tu entorno local
RUTA_TESSERACT = r"C:/Program Files/Tesseract-OCR"
ARCHIVO_PDF = r"C:/Users/Edward/Downloads/20251209-V-30-109.pdf"
ARCHIVO_SALIDA = "tablas_extraidas_corregido.xlsx"

# Nombres estándar para las columnas (para evitar desfases)
COLUMNAS_ESTANDAR = [
    "CÓDIGO", 
    "DESCRIPCIÓN", 
    "UNIDAD", 
    "CUOTA_INICIATIVA", 
    "CUOTA_DICTAMEN"
]

# ==========================================
# 2. FUNCIONES DE LIMPIEZA
# ==========================================

def configurar_entorno():
    if os.path.exists(RUTA_TESSERACT):
        os.environ["PATH"] += os.pathsep + RUTA_TESSERACT
        return True
    else:
        print(f"❌ ERROR: No se encuentra Tesseract en: {RUTA_TESSERACT}")
        return False

def limpiar_texto(val):
    """Limpia saltos de línea y espacios extra."""
    if pd.isna(val):
        return val
    return str(val).replace('\n', ' ').strip()

def intentar_separar_cuotas(row):
    """
    Si la columna de Dictamen está vacía pero la de Iniciativa tiene dos números
    (ej: '35 25'), intenta separarlos.
    """
    iniciativa = str(row['CUOTA_INICIATIVA']).strip()
    dictamen = str(row['CUOTA_DICTAMEN']).strip()
    
    # Si dictamen está vacío (o es nan/None) y la iniciativa parece tener dos valores
    if (dictamen == 'nan' or dictamen == 'None' or dictamen == '') and ' ' in iniciativa:
        partes = iniciativa.split()
        # Verificamos si parece que son dos números (ej. '10 7' o '35 25')
        if len(partes) >= 2:
            # Asumimos que el último valor es el dictamen
            nuevo_dictamen = partes[-1]
            nueva_iniciativa = " ".join(partes[:-1])
            return nueva_iniciativa, nuevo_dictamen
            
    return row['CUOTA_INICIATIVA'], row['CUOTA_DICTAMEN']

def estandarizar_tabla(df, num_pagina):
    """
    Limpia y estandariza una tabla individual antes de concatenarla.
    """
    # 1. Eliminar filas vacías
    df = df.dropna(how='all')
    
    # 2. Eliminar columnas que sean totalmente vacías (bordes fantasma)
    df = df.dropna(axis=1, how='all')
    
    # 3. Normalizar a 5 columnas
    # Si hay más de 5, conservamos las 5 primeras (asumiendo orden izquierda-derecha)
    # Si hay menos, añadimos vacías.
    columnas_actuales = df.shape[1]
    
    if columnas_actuales > 5:
        # A veces crea columnas extra al final vacías o con basura
        df = df.iloc[:, :5]
    elif columnas_actuales < 5:
        # Rellenar con columnas vacías hasta llegar a 5
        for _ in range(5 - columnas_actuales):
            df[len(df.columns)] = None

    # 4. Asignar nombres estándar
    df.columns = COLUMNAS_ESTANDAR
    
    # 5. Detectar y eliminar fila de encabezado si existe en los datos
    # Buscamos si la primera fila contiene "CÓDIGO" o "DESCRIPCIÓN"
    if not df.empty:
        fila_0 = " ".join([str(x).upper() for x in df.iloc[0].values])
        if "CÓDIGO" in fila_0 or "DESCRIPCIÓN" in fila_0:
            df = df.iloc[1:] # Eliminar primera fila
    
    # 6. Añadir metadato de página
    df['Pagina_Origen'] = num_pagina + 1
    
    return df

# ==========================================
# 3. EXTRACCIÓN PRINCIPAL
# ==========================================

def extraer_tablas(ruta_pdf):
    try:
        ocr = TesseractOCR(n_threads=1, lang="spa")
    except Exception as e:
        print(f"❌ Error OCR: {e}")
        return None

    print(f"🔄 Leyendo PDF: {ruta_pdf}...")
    doc = PDF(src=ruta_pdf)

    print("⏳ Ejecutando OCR (esto puede tardar)...")
    try:
        # borderless_tables=False para usar las líneas de la tabla
        tablas_extraidas = doc.extract_tables(ocr=ocr,
                                              implicit_rows=False,
                                              borderless_tables=False,
                                              min_confidence=50)
    except Exception as e:
        print(f"❌ Error extrayendo tablas: {e}")
        return None

    lista_dfs = []

    for numero_pagina, tablas in tablas_extraidas.items():
        for tabla in tablas:
            df = tabla.df
            
            # Procesar la tabla individualmente
            df_limpio = estandarizar_tabla(df, numero_pagina)
            
            # Solo añadir si tiene datos
            if not df_limpio.empty:
                lista_dfs.append(df_limpio)

    if lista_dfs:
        print(f"✅ Se encontraron {len(lista_dfs)} tablas. Concatenando...")
        
        # Concatenación segura (ahora todos tienen las mismas columnas)
        df_final = pd.concat(lista_dfs, ignore_index=True)
        
        # ==========================================
        # 4. LIMPIEZA POST-CONCATENACIÓN
        # ==========================================
        
        # A. Limpieza de texto general
        for col in COLUMNAS_ESTANDAR:
            df_final[col] = df_final[col].apply(limpiar_texto)
            
        # B. Eliminar filas que sean encabezados repetidos (basura residual)
        # Filtramos filas donde la columna CÓDIGO diga literalmente "CÓDIGO" o similar
        filtro_basura = df_final['CÓDIGO'].str.upper().str.contains("CÓDIGO|DESCRIPCIÓN", na=False)
        df_final = df_final[~filtro_basura]
        
        # C. Eliminar filas donde el CÓDIGO sea demasiado corto (ruido OCR)
        df_final = df_final[df_final['CÓDIGO'].str.len() > 3]
        
        # D. Corregir "Huecos" en las cuotas (Separar valores fusionados)
        # Aplica la lógica fila por fila
        df_final[['CUOTA_INICIATIVA', 'CUOTA_DICTAMEN']] = df_final.apply(
            lambda x: pd.Series(intentar_separar_cuotas(x)), axis=1
        )

        return df_final
    else:
        return None

# ==========================================
# 4. EJECUCIÓN
# ==========================================

if __name__ == "__main__":
    if configurar_entorno():
        if os.path.exists(ARCHIVO_PDF):
            df_resultado = extraer_tablas(ARCHIVO_PDF)
            
            if df_resultado is not None and not df_resultado.empty:
                try:
                    df_resultado.to_excel(ARCHIVO_SALIDA, index=False)
                    print(f"🎉 ¡ÉXITO! Archivo guardado: {ARCHIVO_SALIDA}")
                    print(df_resultado.head())
                    print(f"\nTotal de filas extraídas: {len(df_resultado)}")
                except PermissionError:
                    print(f"❌ ERROR: Cierra el archivo Excel '{ARCHIVO_SALIDA}' antes de correr el código.")
            else:
                print("⚠️ No se pudo extraer información válida.")
        else:
            print(f"❌ No encuentro el archivo: {ARCHIVO_PDF}")

🔄 Leyendo PDF: C:/Users/Edward/Downloads/20251209-V-30-109.pdf...
⏳ Ejecutando OCR (esto puede tardar)...
✅ Se encontraron 78 tablas. Concatenando...
🎉 ¡ÉXITO! Archivo guardado: tablas_extraidas_corregido.xlsx
       CÓDIGO                                     DESCRIPCIÓN UNIDAD  \
1  3303.00.01                               Aguas de tocador.   None   
2  3303.00.99                                      Los demás.   None   
3   3304.10.0  Preparaciones para el maquillaje de os labios.     Kg   
4   3304.20.0   Preparaciones para el maquillaje de los ojos.     Kg   
5   3304.30.0       Preparaciones para manicuras o pedicuros.     Kg   

  CUOTA_INICIATIVA CUOTA_DICTAMEN  Pagina_Origen  
1               35             25              1  
2               35             25              1  
3               50             36              1  
4               35             25              1  
5               35             25              1  

Total de filas extraídas: 1438
